# Data profiling

The purpose of this notebook is to perform a comprehensive audit of the **IEEE-CIS Fraud Detection dataset(Vesta)**. In an ML production environment, this step is critical: it enables the transition from simple data ingestion to a deep understanding of business signals.

This assessment follows a 8-step “Zero-Trust” methodology aimed at:
* Ensure the structural integrity of the data (uniqueness, volume, completeness).
* Qualify variables to guide feature engineering strategies (encoding, imputation, outlier handling).
* Identify fraud patterns through statistical analysis and the detection of outliers.
* Detect risks of model drift prior to the supervised modeling phase.

Following this analysis, the data will be deemed “ML-ready” for the construction of the **Silver layer**.



## 1. Volumetry & Target

Objective: To confirm that the Bronze dataset has been fully ingested and to measure the actual fraud rate. The actual fraud rate determines the choice of evaluation metric for our model (AUC-PR) 
* How many rows are in each table? Do they match the source files?
* What is the exact fraud rate?
* Is the target truly binary (0/1), with no NULL values?

 



### a-Number of rows and columns of each dataset

In [0]:
%sql
-- Number of rows in each table
-- Counting the number of rows in each table
SELECT 'transactions_raw' AS table_name, COUNT(*) AS nb_rows FROM fraud.bronze.transactions_raw
UNION ALL
SELECT 'identity_raw', COUNT(*) FROM fraud.bronze.identity_raw;

table_name,nb_rows
transactions_raw,590540
identity_raw,144233


In [0]:
%sql
-- number of columns in transaction table
DESCRIBE TABLE fraud.bronze.transactions_raw;

col_name,data_type,comment
TransactionID,int,null
isFraud,int,null
TransactionDT,int,null
TransactionAmt,double,null
ProductCD,string,null
card1,int,null
card2,double,null
card3,double,null
card4,string,null
card5,double,null


In [0]:
%sql
-- Number of columns in identity table
DESCRIBE TABLE fraud.bronze.identity_raw;

col_name,data_type,comment
TransactionID,int,null
id_01,double,null
id_02,double,null
id_03,double,null
id_04,double,null
id_05,double,null
id_06,double,null
id_07,double,null
id_08,double,null
id_09,double,null


In the **transaction table**, we have **394 columns** and **590540 rows**

In the **identity table**, we have **41 columns** and **144233 rows**


### b. fraud rate evaluation

In [0]:
%sql
-- Fraud rate evaluation
WITH fraud_number AS (
SELECT
    COUNT(isFraud) as fraud_nb
FROM fraud.bronze.transactions_raw
WHERE isFraud = 1
)
SELECT
    fraud_nb,
    ROUND(fraud_nb / (SELECT COUNT(*) FROM fraud.bronze.transactions_raw) * 100, 2) as fraud_rate
FROM fraud_number;
    

fraud_nb,fraud_rate
20663,3.5


Our data set is extremely imbalanced with just 3.5% of fraud

In [0]:
%sql
-- Verifying if our target (isFraud) is binary without null values
SELECT 
    isFraud,
    COUNT(*) AS fraud_label
FROM fraud.bronze.transactions_raw
GROUP BY isFraud;

isFraud,fraud_label
1,20663
0,569877


## 2. Completeness (NULL profiling)

Objective: Measure the NULL rate by column to decide the fate of each feature.
* Which columns are 100% NULL (and therefore unusable)?
* Which columns have a high NULL rate but are still informative?
* Which columns are valid?

Decision : Explicit list of columns to drop (100% NULL) vs. columns to keep despite NULL values (tree-based models treat NULL as information). No hard-and-fast rule such as “> 50% → drop.”



In [0]:
# Missing values evaluation in transaction table
from pyspark.sql import functions as F

# 1. Retrieve the list of columns from the catalog
cols = [r.column_name for r in spark.sql(
    "SELECT column_name FROM fraud.information_schema.columns "
    "WHERE table_schema='bronze' AND table_name='transactions_raw' "
    "ORDER BY ordinal_position"
).collect()]

# 2. Build the query (a single pass over the table using the agg CTE)
agg_exprs = ", ".join([f"COUNT(*) - COUNT(`{c}`) AS n_{c}" for c in cols])
stack_args = ", ".join([
    f"'{c}', n_{c}, ROUND(100.0 * n_{c} / total, 2)" for c in cols
])

query = f"""
WITH agg AS (
  SELECT {agg_exprs}, COUNT(*) AS total
  FROM fraud.bronze.transactions_raw
)
SELECT stack({len(cols)}, {stack_args}) AS (column_name, null_count, null_pct)
FROM agg
"""

# 3. Execute and display sorted by NULL rate in descending order
profile_df = spark.sql(query).orderBy(F.col("null_pct").desc())
display(profile_df)

column_name,null_count,null_pct
dist2,552913,93.63
D7,551623,93.41
D13,528588,89.51
D14,528353,89.47
D12,525823,89.04
D6,517353,87.61
D9,515614,87.31
D8,515614,87.31
V161,508595,86.12
V145,508589,86.12


### Results — Completeness of `transactions_raw`

**Volume**: 394 columns audited, 590,540 rows.

**Distribution of NULL Rates**:
- **0 columns with 100% NULL**
- **168 columns between 76.05% and 99.9%** → retained. This cluster shares 
  a nearly identical rate (~76%), suggesting a structured pattern of missing values 
  (filled in only under certain transaction conditions).
  A nullity indicator will be created in Gold.e).
- **111 valid columns (<5% NULL)** → retained.
- **dist1 (59.65%) and dist2 (>70%)** → retain. Columns known to be 
  informative despite the NULL values.
- **R_emaildomain (>70% NULL)** → retain. The NULL values here are likely 
  meaningful (absence of a recipient's email domain).

**Silver Decisions**:
- Retain: all others, including the 168 columns from the 76% cluster.
- To explore in Gold: create binary indicators "is_null_<col>" for 
  columns with high null rates, so that the model can leverage the absence pattern.

**Justification**: The rule “>50% NULL → removal” is not suitable for 
tree-based models (LightGBM, XGBoost), which treat NULL as a 
category. Only 100% NULL is truly uninformative.


In [0]:
# Missing values evaluation in identity table
from pyspark.sql import functions as F

# 1. Retrieve the list of columns from the catalog
cols = [r.column_name for r in spark.sql(
    "SELECT column_name FROM fraud.information_schema.columns "
    "WHERE table_schema='bronze' AND table_name='identity_raw' "
    "ORDER BY ordinal_position"
).collect()]

# 2. Build the query 
agg_exprs = ", ".join([f"COUNT(*) - COUNT(`{c}`) AS n_{c}" for c in cols])
stack_args = ", ".join([
    f"'{c}', n_{c}, ROUND(100.0 * n_{c} / total, 2)" for c in cols
])

query = f"""
WITH agg AS (
  SELECT {agg_exprs}, COUNT(*) AS total
  FROM fraud.bronze.identity_raw
)
SELECT stack({len(cols)}, {stack_args}) AS (column_name, null_count, null_pct)
FROM agg
"""

# 3. Execute and display sorted by NULL rate in descending order
profile_df = spark.sql(query).orderBy(F.col("null_pct").desc())
display(profile_df)

column_name,null_count,null_pct
id_24,139486,96.71
id_25,139101,96.44
id_08,139078,96.43
id_07,139078,96.43
id_23,139064,96.42
id_21,139074,96.42
id_22,139064,96.42
id_27,139064,96.42
id_26,139070,96.42
id_18,99120,68.72


### Results — Completeness of `identity_raw`

**Volume**: 41 columns audited, 144,233 rows.

**Distribution**:
- **X columns with 100% NULL values** → deleted.
- **9 columns with 96.42% NULL values** → coherent cluster (likely the `id_*` 
  fields linked to browser information, which are rarely populated). Decided to retain 
  them after verifying that the non-null value is a distinguishing factor.
- **Subsequent columns**: jump to 68.72%, followed by a gradual decline.

**Silver Decision**: **No columns are removed due to NULL values.** All 41 columns 
are retained. A `has_device_fingerprint` flag (aggregating the 9 `id_*` columns at 
96.42%) will be created in Gold as a single feature.

**Justification**: For `identity`, the presence or absence of an 
`id_*` value directly reflects the device’s ability to capture data—this is 
strong business information, not a data defect.

## Step 3 — Uniqueness & Relational Integrity

Objective: Ensure the reliability of joins and detect duplicates.
* Is the TransactionID unique in each table?
* Are there any TransactionIDs in the identity sequence that are not associated with any transactions (orphaned)?


In [0]:
%sql
-- Checking for duplicates in TransactionID in transactions_raw table
SELECT
    TransactionID,
    COUNT(*)
FROM fraud.bronze.transactions_raw
GROUP BY TransactionID
HAVING COUNT(*) > 1;

TransactionID,COUNT(*)


In [0]:
%sql
-- Checking for duplicates in TransactionID in identity_raw table
SELECT
    TransactionID,
    COUNT(*)
FROM fraud.bronze.identity_raw
GROUP BY TransactionID
HAVING COUNT(*) > 1;

TransactionID,COUNT(*)


In [0]:
%sql
-- Are there any TransactionIDs in the identity table that are not associated with any transactions (orphaned)?
SELECT i.TransactionID
FROM fraud.bronze.identity_raw i
LEFT JOIN fraud.bronze.transactions_raw t ON i.TransactionID = t.TransactionID
WHERE t.TransactionID IS NULL;

TransactionID


### Diagnosis: Relational Integrity and Uniqueness Audit

1. Uniqueness Audit (Primary Key: TransactionID)

**Methodology**: Analysis of the distribution of occurrences by TransactionID using a GROUP BY clause and filtering for duplicates (HAVING COUNT(*) > 1).

**Result**: No multiple occurrences were detected in the `transactions_raw` and `identity_raw` tables.

**Conclusion**: The `TransactionID` column strictly complies with the uniqueness constraint. 

2. Referential Integrity Audit (“Orphan” Transactions)

**Methodology**: Search for records present in the identity table but absent from the transactions table 

**Result**: No orphan transactions detected. Each identity record is correctly linked to a source transaction.

**Conclusion**: The two data sources are fully aligned.


## Step 4 — Cardinality & Quasi-Constant Columns
Objective: Determine the encoding strategy and identify unnecessary columns based on quasi-constancy.
* How many distinct values are there per categorical column?
* Are there any columns with cardinality 1 (a single value in almost every row)?
* Which columns require one-hot encoding, frequency encoding, or target encoding?

Decision made: Constant columns → remove. Low cardinality → one-hot encoding is an option. High cardinality → Gold frequency encoding. Target encoding → only in the ML pipeline (fold-safe), never in Silver.


In [0]:
# Cardinality evaluation

from pyspark.sql import functions as F
import pandas as pd

NUMERIC_TYPES = {"int", "bigint", "double", "float", "long", "short",
                 "decimal", "tinyint"}

CATEGORICAL_IDS = {"card1", "card2", "card3", "card4", "card5", "card6",
                   "addr1", "addr2"}


def profile_cardinality(table_name: str) -> pd.DataFrame:
    df = spark.table(table_name)
    cols = df.columns
    dtypes = dict(df.dtypes)
    n = df.count()

    # 1. A single pass: `countDistinct` for all columns
    exprs = [F.countDistinct(F.col(f"`{c}`")).alias(f"c_{i}")
             for i, c in enumerate(cols)]
    row = df.agg(*exprs).collect()[0]
    cardinalities = {c: row[f"c_{i}"] for i, c in enumerate(cols)}

    rows = []
    for c in cols:
        card = cardinalities[c]
        dtype = dtypes[c]
        pct = round(100.0 * card / n, 4) if n else 0.0

        # 2. Business-specific overrides first (column name)
        if c in CATEGORICAL_IDS:
            ftype = "Categorical (ID)"
            strategy = "Frequency Encoding"
        elif c.startswith("V") and c[1:].isdigit():
            ftype = "Numerical continuous (Vesta)"
            strategy = "Standardization if linear model — nothing for trees"
        elif c.startswith("C") and c[1:].isdigit():
            ftype = "Numerical ordinal (count)"
            strategy = "Standardization if linear model — nothing for trees"
        elif c.startswith("D") and c[1:].isdigit():
            ftype = "Numerical ordinal (delta)"
            strategy = "Standardization if linear model — nothing for trees"
        # 3. dtype Heuristic + Cardinality
        elif dtype in NUMERIC_TYPES and card > 100:
            ftype = "Numerical continuous"
            strategy = "Standardization if linear model — nothing for trees"
        elif card == 1:
            ftype = "Constant"
            strategy = "Suppression (0 variance)"
        elif card <= 20:
            ftype = "Low categorical"
            strategy = "One-Hot Encoding"
        elif card <= 100:
            ftype = "mid categorical"
            strategy = "Frequency Encoding"
        else:
            ftype = "High categorical"
            strategy = ("Frequency Encoding — target encoding prohibited "
                        "in silver (ML pipeline only, fold-safe)")

        rows.append({
            "column_name": c,
            "dtype": dtype,
            "cardinality": card,
            "pct_unique": pct,
            "feature_type": ftype,
            "ml_strategy": strategy,
        })

    pdf = pd.DataFrame(rows)
    pdf = pdf.sort_values("cardinality", ascending=False).reset_index(drop=True)
    return pdf


# --- Execution ---
profile_tx = profile_cardinality("fraud.bronze.transactions_raw")
profile_id = profile_cardinality("fraud.bronze.identity_raw")

# Overview
display(profile_tx.head(50))

# Summary by Feature Type
print("=== transactions_raw ===")
print(profile_tx["feature_type"].value_counts())
print("\n=== identity_raw ===")
print(profile_id["feature_type"].value_counts())

column_name,dtype,cardinality,pct_unique,feature_type,ml_strategy
TransactionID,int,590540,100.0,Numerical continuous,Standardization if linear model — nothing for trees
TransactionDT,int,573349,97.0889,Numerical continuous,Standardization if linear model — nothing for trees
V307,double,37367,6.3276,Numerical continuous (Vesta),Standardization if linear model — nothing for trees
V127,double,24414,4.1342,Numerical continuous (Vesta),Standardization if linear model — nothing for trees
V308,double,23064,3.9056,Numerical continuous (Vesta),Standardization if linear model — nothing for trees
TransactionAmt,double,20902,3.5395,Numerical continuous,Standardization if linear model — nothing for trees
V310,double,19136,3.2404,Numerical continuous (Vesta),Standardization if linear model — nothing for trees
V306,double,16210,2.7449,Numerical continuous (Vesta),Standardization if linear model — nothing for trees
V317,double,15184,2.5712,Numerical continuous (Vesta),Standardization if linear model — nothing for trees
V203,double,14951,2.5318,Numerical continuous (Vesta),Standardization if linear model — nothing for trees


=== transactions_raw ===
feature_type
Numerical continuous (Vesta)    339
Numerical ordinal (delta)        15
Numerical ordinal (count)        14
Low categorical                  11
Catégorical (ID)                  8
Numerical continuous              5
mid categorical                   2
Name: count, dtype: int64

=== identity_raw ===
feature_type
Low categorical         17
mid categorical         12
Numerical continuous     9
High categorical         3
Name: count, dtype: int64


In [0]:
# Persist cardinality profiles in Delta tables
for pdf, name in [(profile_tx, "profile_cardinality_transactions"),
                  (profile_id, "profile_cardinality_identity")]:
    spark.createDataFrame(pdf).write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"fraud.bronze.{name}")

### Cardinality & Encoding Strategy Results

**Method**: a `countDistinct` pass over the 435 columns, classification 
by business-specific overrides (`card*`, `addr*`), followed by a dtype + cardinality heuristic. 
Results stored in `fraud.bronze.profile_cardinality_*`.

**Results for `transactions_raw`**:
- **0 columns with cardinality 1** → no drop due to zero variance.
- 11 columns with low cardinality (2–20) → One-Hot Encoding.
- 2 columns with medium cardinality (21–100) → Frequency Encoding.
- 8 categorical ID columns (`card1-6`, `addr1-2`) → Frequency Encoding.
- 368 numeric columns (`V*`, `C*`, `D*`, `TransactionAmt`, `dist*`) 
  → no encoding; conditional standardization based on the model.

**`identity_raw` results**:
- **0 columns with cardinality 1.**
- 17 low-cardinality categorical, 12 mid-cardinality categorical, 9 numerical, 3 high-cardinality categorical.
- The 3 high-cardinality columns (`DeviceInfo`, `id_31`, `id_33`) 
  → Frequency Encoding (never one-hot).

**Silver Decisions**:
- **No removal** (no constants, no 100% NULL).
- **No encoding performed in Silver.** Encoding (one-hot, frequency) is 
  a transformation in the ML pipeline, applied fold by fold. Target encoding 
  is excluded from Silver (risk of leakage).
- The lists by strategy are read from `fraud.bronze.profile_cardinality_*`.


## 5. — Distribution of `TransactionAmt`

**Méthode** : Descriptive statistics by class for `TransactionAmt` only. 
A comprehensive EDA analysis of all numeric columns will be conducted in a 
dedicated notebook (`03_eda`). The goal here is to determine whether a transformation is necessary 
in Gold.

**Résultats** :

| Statistic | Fraud (n=20 663) | No-fraud (n=569 877) |
|---|---|---|
| min | 0,29 | 0,25 |
| p25 | 35,0 | 44,0 |
| **median** | **75,0** | **68,5** |
| p75 | 161,0 | 120,0 |
| max | 5 191 | 31 937 |
| mean | 149,2 | 134,5 |

**Observations**:
1. **Overall fraud rate = 3.499%** — consistent with expectations.
2. **Fraud transactions have a higher median and P75** (+9.5% and +34%), 
   but a **lower P25** (−20%). Classic signature of *card testing*: 
   test micro-transactions prior to the actual fraud.
3. **The maximum fraud amount is 6× lower** than that of non-fraud transactions (5,191 vs. 
   31,937). Very high-value transactions are almost always 
   legitimate or intercepted upstream.
4. **The average is misleading** (an apparent difference of only 11%) because it is dominated 
   by legitimate outliers. **The median and percentiles are the 
   robust statistics that should be used instead.**


**Gold Decisions**:
- **No mandatory transformations for trees.** LightGBM/XGBoost are 
  invariant to monotonicity; a log transformation adds no value.
- **For linear models**: log1p(`TransactionAmt`) (distribution 
  heavily skewed to the right).



In [0]:
%sql
SELECT 
    isFraud,
    COUNT(*) as count,
    MIN(TransactionAmt) as min_amt,
    PERCENTILE(TransactionAmt, 0.25) as p25_amt,
    PERCENTILE(TransactionAmt, 0.50) as median_amt,
    PERCENTILE(TransactionAmt, 0.75) as p75_amt,
    MAX(TransactionAmt) as max_amt,
    AVG(TransactionAmt) as avg_amt
FROM fraud.bronze.transactions_raw
GROUP BY isFraud;

isFraud,count,min_amt,p25_amt,median_amt,p75_amt,max_amt,avg_amt
1,20663,0.292,35.044,75.0,161.0,5191.0,149.24477858975038
0,569877,0.251,43.97,68.5,120.0,31937.391,134.5116645785027


## Step 6 — Frequent Values & Sentinel Values

Objective: Detect disguised technical values that contaminate categorical features.

Questions it answers:
* What are the dominant values in each categorical column?
* Are there any outlier values (Unknown, -1, 999, NaN as a string)?
* Are there any extremely rare categories that should be grouped together?



In [0]:
# Sentinels checking
from pyspark.sql import functions as F

SENTINEL_STRINGS = ["unknown", "nan", "null", "", "none", "n/a", "anonymous", "notfound"]

def scan_categorical_sentinels(table, columns, top_n=10):
    df = spark.table(table)
    n = df.count()
    for c in columns:
        print(f"\n{'='*50}\n{c}\n{'='*50}")
        # Top N
        (df.groupBy(c).count()
           .orderBy(F.desc("count"))
           .limit(top_n)
           .show(truncate=False))
        # Sentinel detection (case-insensitive)
        sentinel_count = df.filter(
            F.lower(F.col(f"`{c}`").cast("string")).isin(SENTINEL_STRINGS)
        ).count()
        if sentinel_count > 0:
            print(f"Sentinels Detected : {sentinel_count} "
                  f"({100.0 * sentinel_count / n:.2f} %)")


# Transactions
scan_categorical_sentinels(
    "fraud.bronze.transactions_raw",
    ["ProductCD", "card4", "card6", "P_emaildomain", "R_emaildomain",
     "M1", "M2", "M3", "M4", "M5", "M6", "M7", "M8", "M9"]
)

# Identity
scan_categorical_sentinels(
    "fraud.bronze.identity_raw",
    ["DeviceType", "DeviceInfo", "id_12", "id_15", "id_16",
     "id_28", "id_29", "id_30", "id_31", "id_33", "id_34",
     "id_35", "id_36", "id_37", "id_38"]
)


ProductCD
+---------+------+
|ProductCD|count |
+---------+------+
|W        |439670|
|C        |68519 |
|R        |37699 |
|H        |33024 |
|S        |11628 |
+---------+------+


card4
+----------------+------+
|card4           |count |
+----------------+------+
|visa            |384767|
|mastercard      |189217|
|american express|8328  |
|discover        |6651  |
|NULL            |1577  |
+----------------+------+


card6
+---------------+------+
|card6          |count |
+---------------+------+
|debit          |439938|
|credit         |148986|
|NULL           |1571  |
|debit or credit|30    |
|charge card    |15    |
+---------------+------+


P_emaildomain
+-------------+------+
|P_emaildomain|count |
+-------------+------+
|gmail.com    |228355|
|yahoo.com    |100934|
|NULL         |94456 |
|hotmail.com  |45250 |
|anonymous.com|36998 |
|aol.com      |28289 |
|comcast.net  |7888  |
|icloud.com   |6267  |
|outlook.com  |5096  |
|msn.com      |4092  |
+-------------+------+


R_e

### Frequent Values & Sentinel Values

**Method**: Top 10 values per categorical column, 
case-insensitive detection of textual sentinels (`unknown`, `nan`, `anonymous`, 
`notfound`…), and scanning for numeric placeholders (`0`, `-1`). 
The results are interpreted by cross-referencing occurrence **and** business semantics.

**Scope**: 14 categorical columns from `transactions_raw`, 
17 categorical columns from `identity_raw`.

---

#### 1. Categorical columns — `transactions_raw`

**`ProductCD`** — 5 values, no sentinel
| Value | n | % |
|---|---|---|
| W | 439 670 | 74,45 % |
| C | 68 519 | 11,60 % |
| R | 37 699 | 6,38 % |
| H | 33 024 | 5,59 % |
| S | 11 628 | 1,97 % |

→ **No action.** 

**`card4`** — 4 networks + NULL.
| Value | n | % |
|---|---|---|
| visa | 384 767 | 65,15 % |
| mastercard | 189 217 | 32,04 % |
| american express | 8 328 | 1,41 % |
| discover | 6 651 | 1,13 % |
| NULL | 1 577 | 0,27 % |

→ **NULL retained** (0.27% = map not filled in, separate status).

**`card6`** — 3 main values + 2 ultra-rares.
| Valeur | n | % |
|---|---|---|
| debit | 439 938 | 74,49 % |
| credit | 148 986 | 25,23 % |
| NULL | 1 571 | 0,27 % |
| debit or credit | **30** | **0,005 %** |
| charge card | **15** | **0,003 %** |

→ **NULL retained.** `debit or credit` and `charge card` (< 0.01%) are grouped 
under `other` in the ML pipeline (depends on the fold train → excluded from Silver).

**`P_emaildomain`** — Strong prevalence of web-based email services + business category.
| Value | n | % |
|---|---|---|
| gmail.com | 228 355 | 38,67 % |
| yahoo.com | 100 934 | 17,09 % |
| NULL | 94 456 | 16,00 % |
| hotmail.com | 45 250 | 7,66 % |
| **anonymous.com** | **36 998** | **6,26 %** |
| aol.com | 28 289 | 4,79 % |
| comcast.net | 7 888 | 1,34 % |
| icloud.com | 6 267 | 1,06 % |
| outlook.com | 5 096 | 0,86 % |
| msn.com | 4 092 | 0,69 % |

→ **`anonymous.com` is a business category, NOT a sentinel.** 6.26% 
of transactions use an anonymous email address on the buyer’s side—this is a 
known fraud indicator and should be retained as is. **DO NOT map to NULL.**
→ **NULL retained** (email not provided = distinct state).
→ Long tail (domains < 0.5%) → grouped under `other` in the 
ML pipeline.

**`R_emaildomain`** — A nearly identical pattern, but with 76.75% NULL values.
| Value | n | % |
|---|---|---|
| **NULL** | **453 249** | **76,75 %** |
| gmail.com | 57 147 | 9,68 % |
| hotmail.com | 27 509 | 4,66 % |
| **anonymous.com** | **20 529** | **3,48 %** |
| yahoo.com | 11 842 | 2,01 % |
| aol.com | 3 701 | 0,63 % |
| outlook.com | 2 507 | 0,42 % |
| comcast.net | 1 812 | 0,31 % |
| yahoo.com.mx | 1 508 | 0,26 % |
| icloud.com | 1 398 | 0,24 % |

→ **Same logic as `P_emaildomain`**: `anonymous.com` (3.48%) retained 
as a business category. NULL retained (absence of recipient domain = 
information in itself; see step 2).

---

#### 2. Columns `M1` – `M9` — “merchant” status

**Important correction**: Contrary to the initial assumption, **no 
`M*` column contains the text sentinel `“Unknown”`**. The 
`M*` columns have only 2 or 3 states: `T`, `F`, `NULL` (and `M0`/`M1`/`M2` for `M4`). 
**Here, `NULL` is the only “not specified” state**, and it carries meaning 
(the merchant did not report the information).

| Column | Values (top) | NULL % | Reading |
|---|---|---|---|
| M1 | T 54,1 % / F 0,004 % | 45,9 % | `F` nearly nonexistent (25 rows) |
| M2 | T 48,3 % / F 5,75 % | 45,9 % | NULL = M1 NULL (same 271 100) |
| M3 | T 42,6 % / F 11,5 % | 45,9 % | NULL = M1 NULL (same 271 100) |
| M4 | M0 33,3 % / M2 10,1 % / M1 8,9 % | 47,7 % | Merchant codes, no T/F |
| M5 | F 22,4 % / T 18,2 % | 59,3 % | — |
| M6 | F 38,6 % / T 32,7 % | 28,7 % | NULL: the lowest of the M* |
| M7 | F 35,8 % / T 5,57 % | 58,6 % | NULL = M8 NULL (346 252) |
| M8 | F 26,3 % / T 15,1 % | 58,6 % | NULL = M7 NULL (346 252) |
| M9 | T 34,8 % / F 6,54 % | 58,6 % | NULL = M7 NULL (346 252) |

→ **Major structural pattern**: `M1`, `M2`, `M3` share **exactly** 
271,100 NULL values. `M7`, `M8`, `M9` share **exactly** 346,252 NULL values. These are 
not random omissions; rather, it is a **block of features that appeared or 
disappeared together** depending on the time period or merchant.
→ **Silver Decision**: Keep `T`, `F`, `NULL` as is. In the Gold model, 
create flags `m_block1_missing` = `(M1 IS NULL)` and `m_block3_missing` = 
`(M7 IS NULL)` to capture this structural pattern.

---

#### 3. Categorical columns — `identity_raw`

**`DeviceType`** — 3 states.
| Value | n | % |
|---|---|---|
| desktop | 85 165 | 59,04 % |
| mobile | 55 645 | 38,58 % |
| NULL | 3 423 | 2,37 % |

→ **No action.**

**`DeviceInfo`** — High cardinality, long tail of OS/browser combinations.
| Value | n | % |
|---|---|---|
| Windows | 47 722 | 33,09 % |
| NULL | 25 567 | 17,73 % |
| iOS Device | 19 782 | 13,72 % |
| MacOS | 12 573 | 8,72 % |
| Trident/7.0 | 7 440 | 5,16 % |
| rv:11.0 | 1 901 | 1,32 % |
| … (hundreds of device-specific variants) | | |

→ **Feature Engineering Gold**: Extract a normalized OS family 
(`Windows`, `iOS`, `macOS`, `Android`, `Linux`, `Trident`, `other`) to 
reduce cardinality. Keep raw `DeviceInfo` for frequency encoding.

---

#### 4. `id_*` columns — sentinels detected: **4 false positives**

The script reported `NotFound` or `Unknown` as a sentinel for 
`id_12`, `id_15`, `id_16`, and `id_29`. **These are false positives**: these 
values are not technical placeholders; they are **legitimate business categories**, 
often dominant ones.

| Column | Value | Occurrence | Verdict |
|---|---|---|---|
| `id_12` | `NotFound` | 123,025 (**85.30%**) | ❌ False positive — dominant status |
| `id_15` | `Unknown` | 11,645 (**8.07%**) | ❌ False positive — business status |
| `id_16` | `NotFound` | 63,016 (**43.69%**) | ❌ False positive — dominant status |
| `id_29` | `NotFound` | 66,052 (**45.80%**) | ❌ False positive — dominant status |

**Details of columns with 3 business states (Found / NotFound / NULL)**:

| Column | Statuses | Read |
|---|---|---|
| id_12 | NotFound 85.3% / Found 14.7% / 0 NULL | Default status = NotFound |
| id_15 | Found 46.96% / New 42.72% / Unknown 8.07% / NULL 2.25% | `Unknown` = explicit status |
| id_16 | Found 45.98% / NotFound 43.69% / NULL 10.32% | Balanced distribution |
| id_28 | Found 52.85% / New 44.89% / NULL 2.26% | No sentinel |
| id_29 | Found 51.95% / NotFound 45.80% / NULL 2.26% | Balanced distribution |

→ **No values mapped to NULL.** `NotFound` and `Unknown` are 
business categories that encode the status of the device verification—these are 
precisely the signals that fraud models rely on. Mapping them to 
NULL would destroy the signal.

**`id_34`** — A true technical placeholder has been detected.
| Value | n | % |
|---|---|---|
| NULL | 66 428 | 46,05 % |
| match_status:2 | 60 011 | 41,61 % |
| match_status:1 | 17 376 | 12,05 % |
| match_status:0 | 415 | 0,29 % |
| **match_status:-1** | **3** | **0,002 %** |

→ **The only true outlier in the set**: `match_status:-1` (3 lines, 0.002%). 
Semantically, this is a system error code, not a matching status. 
**Map `match_status:-1` → `NULL`** in Silver, and group the rest of the 
`match_status:*` entries into retained categories.

**`id_35` – `id_38`** — Boolean T/F flags with a common structural NULL.
| Colonne | T | F | NULL |
|---|---|---|---|
| id_35 | 77 814 (53,95 %) | 63 171 (43,80 %) | 3 248 (2,25 %) |
| id_36 | 6 919 (4,80 %) | 134 066 (92,95 %) | 3 248 (2,25 %) |
| id_37 | 110 452 (76,58 %) | 30 533 (21,17 %) | 3 248 (2,25 %) |
| id_38 | 67 063 (46,50 %) | 73 922 (51,25 %) | 3 248 (2,25 %) |

→ **Structural pattern**: The 4 columns share **exactly** 3,248 NULL values. 
Same logic as `M1-M3` / `M7-M9`: a block of features missing together.
→ **Silver decision**: Keep T/F/NULL. Create a flag in Gold 
`id_35_38_missing` = `(id_35 IS NULL)`.

---

#### 5. Ultra-rare categories to group together

| Column | Value | n | % | Decision |
|---|---|---|---|---|
| `card6` | `debit or credit` | 30 | 0.005% | Group under `“other”` (ML pipeline) |
| `card6` | `charge card` | 15 | 0.003% | Group under `“other”` (ML pipeline) |
| `id_34` | `match_status:-1` | 3 | 0.002% | **Map to NULL** in Silver |
| `id_34` | `match_status:0` | 415 | 0.29% | Keep (explicit status) |

Le regroupement en `"other"` se fait **dans le pipeline ML, pas en Silver** : 
la liste des catégories rares dépend du fold d'entraînement (une catégorie 
rare sur le train peut être absente en validation → leakage de fréquence).

---

#### 6. Structural Patterns of NULL Values to Leverage

Three column blocks share **exactly** the same number of NULL values, 
which reveals a common pattern:

| Block | Columns | Shared NULLs | % |
|---|---|---|---|
| Merchant Block 1 | M1, M2, M3 | 271,100 | 45.9% |
| Merchant Block 2 | M7, M8, M9 | 346,252 | 58.6% |
| Device flags | id_35, id_36, id_37, id_38 | 3,248 | 2.25% |

→ **Gold Candidate Features** (to be validated by the fraud rate):
- `m_block1_missing` = `(M1 IS NULL)`
- `m_block2_missing` = `(M7 IS NULL)`
- `device_flags_missing` = `(id_35 IS NULL)`

These flags will transform a grouped absence pattern into a single binary feature, 
which is often more discriminatory than 3 or 9 individual columns.

---

### Silver Decisions — Summary

**No `NotFound` / `Unknown` → NULL mappings.** These are 
dominant business categories (`id_12`: 85%, `id_16`: 44%, `id_29`: 46%). 
Automatic detection flagged them incorrectly.

**No `anonymous.com` → NULL mappings** on `P_emaildomain` (6.26%) and 
`R_emaildomain` (3.48%). This is a business category associated with a high 
risk of fraud.

**No mapping of `NULL` to a default value.** NULL values in `M*`, `id_*`, 
`dist*`, and `email` are for informational purposes only. LightGBM/XGBoost handle them natively.

**Case normalization** using `lower()` on text columns: 
`DeviceInfo`, `id_30`, `id_31`, `id_33`, and email addresses. This prevents 
category duplicates (`Windows` vs. `windows`).

**Only one true sentinel mapped**: `id_34 = “match_status:-1”` → `NULL` 
(3 lines, system error code).

**Groupings under `“other”` passed to the ML pipeline**: categories 
< 0.01% (`card6`: `debit or credit`, `charge card`) and the long tail 
of emails. Depends on the train fold; no Silver.

**Gold Candidate Features**: 3 block flags (`m_block1_missing`, 
`m_block2_missing`, `device_flags_missing`) + 2 issues from steps 5–6 
(`is_null_dist1`, `address_mismatch_flag`).


## 7. Business Validity (Hard Rules)

Objective: Verify that the data complies with the basic business constraints.
* Are there any negative amounts (anomalies)?
* Is the distribution of revenue consistent?
* Are the hard logical constraints satisfied?


In [0]:
%sql
-- A. Negative amounts
SELECT 'negative_amt' AS check_name, COUNT(*) AS n
FROM fraud.bronze.transactions_raw
WHERE CAST(TransactionAmt AS DOUBLE) < 0;



check_name,n
negative_amt,0


In [0]:
%sql
-- B. TransactionDT (date) consistent
SELECT 'dt_range' AS check_name,
  MIN(CAST(TransactionDT AS BIGINT)) AS dt_min,
  MAX(CAST(TransactionDT AS BIGINT)) AS dt_max
FROM fraud.bronze.transactions_raw;



check_name,dt_min,dt_max
dt_range,86400,15811131


In [0]:
%sql
-- C. ProductCD Breakdown  
SELECT ProductCD, COUNT(*) AS n,
  ROUND(100.0 * SUM(isFraud) / COUNT(*), 3) AS pct_fraude
FROM fraud.bronze.transactions_raw
GROUP BY ProductCD ORDER BY pct_fraude DESC;

ProductCD,n,pct_fraude
C,68519,11.687
S,11628,5.900
H,33024,4.766
R,37699,3.783
W,439670,2.040


### Business Validity

**A. Negative amounts**: 0. → No anomalies in `TransactionAmt`.

**B. Time Range of `TransactionDT`**:
- `dt_min` = 86,400 (exactly 1 day in seconds)
- `dt_max` = 15,811,131
- Range ≈ 182 days (6 months)

`TransactionDT` is indeed a delta in seconds from an unknown reference point
(not an absolute timestamp). The minimum value of 86,400 = 1 day
suggests that the dataset starts on day 1, not day 0.

**C. Breakdown of Products and Fraud Rates**:

| ProductCD | n | % of dataset | fraud_rate | vs. baseline (3.499%) |
|---|---|---|---|---|
| C | 68,519 | 11.60% | **11.687%** | **×3.34**|
| S | 11,628 | 1.97% | **5.900%** | ×1.69 |
| H | 33,024 | 5.59% | **4.766%** | ×1.36 |
| R | 37,699 | 6.38% | **3.783%** | ×1.08 |
| W | 439,670 | 74.45% | **2.040%** | ×0.58 |

**Observations**:
1. **`C` (cash-out) accounts for 11.69% of fraud, which is 3.3 times the baseline rate.**
   This is the riskiest product in the dataset. It accounts for only 11.6% of
   transactions but contributes to ~38% of fraud.
2. **`W` (the most common, 74.5%)** is the least fraudulent (2.04%).
   This represents the “normal behavior” of the dataset—dominated by traditional
   online purchases.
3. **The fraud rate varies by a factor of 5.7× between `C` and `W`.** `ProductCD`
   is therefore a first-order signal.

**Silver Decision**: `ProductCD` retained as-is (5 categories, low
cardinality → One-Hot in the ML pipeline).

**Gold Decision**: two candidate features:
- `is_high_risk_product` = `(ProductCD IN (‘C’, ‘S’))`
- `product_fraud_rate` — target encoding **to be performed only in the ML pipeline**
  (fold-safe), never in Silver.

**Validation**: No business anomalies detected. The dataset is valid
with respect to the hard invariants (amount ≥ 0, binary target, consistent
time range).

## 8. Temporal Analysis & Drift

Objective: To understand the temporal dimension and determine whether the fraud rate drifts over time.
*What is the time range of the dataset (TransactionDT = delta in seconds, not a timestamp)?
* Is the fraud rate stable over time, or does it vary?
* Does the distribution of features change between the beginning and end of the period?


In [0]:
%sql
-- Fraud rate by day
SELECT
  FLOOR(TransactionDT / 86400) AS day_num,
  COUNT(*) AS n,
  SUM(isFraud) AS fraud,
  ROUND(100.0 * SUM(isFraud) / COUNT(*), 3) AS pct_fraud
FROM fraud.bronze.transactions_raw
GROUP BY FLOOR(TransactionDT / 86400)
ORDER BY day_num;

day_num,n,fraud,pct_fraud
1,5122,112,2.187
2,3730,123,3.298
3,3241,92,2.839
4,4036,115,2.849
5,3964,127,3.204
6,3717,99,2.663
7,3786,136,3.592
8,3999,93,2.326
9,3907,119,3.046
10,3445,90,2.612


In [0]:
%sql
--Drift
WITH periodes AS (
  SELECT *,
    CASE WHEN TransactionDT <  8000000 THEN 'early'
         WHEN TransactionDT >= 8000000 THEN 'late'
         ELSE 'mid' END AS periode
  FROM fraud.bronze.transactions_raw
)
SELECT
  periode,
  COUNT(*) AS n,
  ROUND(100.0 * AVG(isFraud), 3) AS pct_fraude,
  ROUND(AVG(CAST(TransactionAmt AS DOUBLE)), 2) AS avg_amt,
  ROUND(PERCENTILE(CAST(TransactionAmt AS DOUBLE), 0.5), 2) AS median_amt,
  ROUND(100.0 * SUM(CASE WHEN P_emaildomain = 'anonymous.com' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_anonymous_email,
  ROUND(100.0 * SUM(CASE WHEN dist1 IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_null_dist1
FROM periodes
GROUP BY periode
ORDER BY periode;

periode,n,pct_fraude,avg_amt,median_amt,pct_anonymous_email,pct_null_dist1
early,321338,3.405,133.5,68.95,6.72,61.88
late,269202,3.611,136.85,68.5,5.72,57.00


### Temporal Analysis & Drift

**Time range**: `dt_min` = 86,400 (day 1), `dt_max` = 15,811,131 (day 182). Range = **182 days** (6 months). `TransactionDT` is a delta in seconds, never an absolute timestamp.
**Fraud Rate Trend — Endpoints**:
- Day 1: **2.187%** (112 fraud cases / 5,122 transactions)
- Day 182: **4.321%** (119 frauds / 2,754 transactions)
- Ratio: **×1.98** (nearly doubled)

**Fraud Rate Trend — Early/Late Aggregates**:
| Period | n | fraud_pct |
|---|---|---|
| early (days 1–92) | 321,338 | **3.405%** |
| late (days 93–182) | 269,202 | **3.611%** |

**Interpretation**: The drift is **real but non-monotonic**. The endpoints suggest a doubling (×1.98), but the early/late aggregates show only a 0.2-point increase (+6% in relative terms). This means that **the fraud rate fluctuates significantly over the period**: some days are exceptionally high (end of the period), others low (beginning). The trend is volatile, not linear.

**Feature Drift**:
| Feature | early | late | Difference | Relative |
|---|---|---|---|---|
| avg_amt | 133.50 | 136.85 | +3.35 | +2.5% |
| median_amt | 68.95 | 68.50 | −0.45 | −0.7% |
| pct_anonymous_email | 6.72% | 5.72% | −1.00 pt | **−14.9%** |
| pct_null_dist1 | 61.88% | 57.00% | −4.88 pts | **−7.9%** |

**Observations**:
1. The fraud rate **varies significantly from day to day** (ranging from 2.19% to 4.32% at the extremes), even though the aggregate trend over 6 months is moderate (+6%).
2. The categorical/structural features **have shifted significantly**: `pct_null_dist1` has dropped by 4.88 points (−7.9% relative), and `pct_anonymous_email` has dropped by 1 point (−14.9% relative). These shifts reflect changes in data collection patterns (device fingerprint, email domain) over time.

**Gold Decision — Mandatory Time Validation**:
- **Random split excluded.**
- **Selected strategy**: chronological split into `TransactionDT` blocks. Example: train = days 1–127 (70%), valid = days 128–154 (15%), test = days 155–182 (15%). Each fold never sees future transactions.

- **Quantitative justification**: the day-to-day variance (2.19% → 4.32%) **and** the feature drift (`pct_null_dist1` −7.9%, `pct_anonymous_email` −14.9%) ensure that a random split would introduce an optimistic bias: The model would see end-of-period transactions during training that it should never have seen, and would evaluate transactions that share their collection artifacts with the train. Temporal validation ensures that the metrics measure the **ability to generalize to the future**.
**MLOps Action**: Add drift monitoring in production (Evidently AI or equivalent). Compare the distribution of features in production with that of the training set; trigger an alert if `null_dist1` drifts by > 10% or if `pct_anonymous_email` drifts by > 20%.

**Limitation**: The analysis focuses on endpoints and aggregates, not on the entire 182-day period. The `03_eda` notebook will supplement this with a day-by-day visualization and a breakdown by `ProductCD`.

